# StockRoute 30 秒体验 · A股数据免费档

**运行前**:左侧菜单 🚩 或顶部输入框设置环境变量 `STOCKROUTE_TOKEN`([免费注册](https://m-stock.600044.xyz) →「Token」页签发)。
免费档:200 次/日,本 notebook 全程约消耗 10 次。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stock-Route/stockroute-terminal/blob/main/examples/quickstart_colab.ipynb)

In [ ]:
%pip install -q stockroute matplotlib
import os
from getpass import getpass
os.environ.setdefault('STOCKROUTE_TOKEN', getpass('粘贴你的 token(输入不回显):'))

from stockroute import StockRoute
sr = StockRoute()
ds = sr.datasets()
print(f"✅ 连接成功!你的档位可见 {ds.get('count')} 个数据集(其中 {ds.get('queryable_count')} 个可直查)")

## 1️⃣ 日 K 线(免费 3 年)
返回 pandas DataFrame,单位口径自动挂载在 `df.attrs['units']`

In [ ]:
k = sr.kline('600519', limit=250)
print(k.tail(3))
print('\n单位口径:', {k2: v for k2, v in list(k.attrs['units'].items())[:4]})

## 2️⃣ 市场宽度(红绿盘家数)
分钟级涨跌家数——市场心跳回放页的同款数据

In [ ]:
import matplotlib.pyplot as plt
import datetime
b = sr.query('sentiment.breadth_minute', limit=200)   # 免费档单次上限 200(全天 241 点需基础档)
b['ts'] = b['ts_ms'].map(lambda m: datetime.datetime.fromtimestamp(int(m)/1000).strftime('%H:%M'))
fig, ax = plt.subplots(figsize=(12,4))
ax.fill_between(b['ts'], b['rise'].astype(int), color='#FF4D5E', alpha=.3, label='上涨家数')
ax.fill_between(b['ts'], -b['fall'].astype(int), color='#2EE6A6', alpha=.3, label='下跌家数')
ax.axhline(0, color='#555', lw=.8); ax.legend()
plt.xticks(b['ts'][::30], rotation=45); plt.title('市场宽度(最新交易日)'); plt.show()

## 3️⃣ 人气榜 & 异动归因(AI 标签)
免费档可查;归因内容由上游 AI 基于公开信息生成,仅供参考

In [ ]:
hot = sr.query('sentiment.hot_ths', limit=10)
print(hot[['rank','name','ticker']].to_string(index=False))

an = sr.query('board.anomaly_reason', limit=3)
for _, x in an.iterrows():
    print(f"\n【{x['tag_name']}】{x['stock_name']}")
    print(' ', x['analysis_content'][:120], '…')

## 下一步
- 全部 96 个数据集:`sr.datasets()` 看清单,或读[接口文档](https://m-stock.600044.xyz/docs)
- 付费集(龙虎榜/筹码分布/分钟线)需[开通档位](https://m-stock.600044.xyz)
- 可视化终端:本仓库 Docker 一键部署